# B05 · Sesión 1 — Primer motor de reglas con `experta`

**Objetivo (RA5-a):** entender hechos, reglas, `DefFacts` y el ciclo *reconocer-actuar* construyendo un motor de diagnóstico con la librería `experta`.

> Práctica guiada de la Sesión 1 de los [apuntes](../apuntes.md). Se trabaja en clase; no se entrega.

In [ ]:
%pip install experta

# Parche obligatorio en Python 3.10+ (collections.Mapping se eliminó)
import collections, collections.abc
if not hasattr(collections, "Mapping"):
    collections.Mapping = collections.abc.Mapping
    collections.Iterable = collections.abc.Iterable
    collections.MutableMapping = collections.abc.MutableMapping

from experta import *

## 1. Hechos: la unidad básica

Un `Fact` es un diccionario. No mantiene orden y puedes usar claves o valores posicionales.

In [ ]:
f = Fact(a=1, b=2)
print(f["a"])

# valores posicionales -> índice numérico
g = Fact("x", "y", "z")
print(g[1])

# mezcla (posicionales primero)
h = Fact("x", "y", z=9)
print(h[0], h["z"])

## 2. Reglas: LHS y RHS

El **LHS** son patrones (condiciones) y el **RHS** es el cuerpo que se ejecuta. `salience` fija la prioridad.

In [ ]:
class DiagnosticoPC(KnowledgeEngine):
    @DefFacts()
    def inicio(self):
        yield Fact(accion="diagnosticar")

    @Rule(Fact(accion="diagnosticar"), salience=10)
    def arrancar(self):
        print("Diagnóstico del PC...")
        self.declare(Fact(luz_encendida=True), Fact(sonido="pitidos_cortos"))

    @Rule(Fact(luz_encendida=True), Fact(sonido="pitidos_cortos"))
    def ram(self):
        self.declare(Fact(causa="problema_ram"))

    @Rule(Fact(causa="problema_ram"))
    def resultado(self):
        print("DIAGNÓSTICO: fallo de memoria RAM.")

motor = DiagnosticoPC()
motor.reset()
motor.run()

## 3. Actividad guiada — triaje de incidencias TI

Completa las reglas para que una incidencia sea `critica` si `impacto="alto"` **o** `usuarios > 50`; `media` si `usuarios > 10`; `baja` en otro caso. Guarda el resultado en un hecho `nivel`.

In [ ]:
class TriajeTI(KnowledgeEngine):
    @DefFacts()
    def _hechos(self):
        yield Fact(incidencia="INC-1")

    # TODO: añade reglas con salience para decidir el nivel
    # Pista: usa P(...) si necesitas comparar un valor (P(lambda u: u > 50))
    ...

# Casos de prueba
for impacto, usuarios in [("alto", 5), ("bajo", 30), ("bajo", 3)]:
    m = TriajeTI()
    m.reset()
    m.declare(Fact(impacto=impacto, usuarios=usuarios))
    m.run()
    print(impacto, usuarios, "->", [dict(f) for f in m.facts.values() if "nivel" in f])

### Solución propuesta

```python
class TriajeTI(KnowledgeEngine):
    @DefFacts()
    def _hechos(self):
        yield Fact(incidencia="INC-1")

    @Rule(OR(Fact(impacto="alto"), Fact(usuarios=P(lambda u: u > 50))), salience=30)
    def critica(self):
        self.declare(Fact(nivel="critica"))

    @Rule(NOT(Fact(nivel=W())), Fact(usuarios=P(lambda u: u > 10)), salience=20)
    def media(self):
        self.declare(Fact(nivel="media"))

    @Rule(NOT(Fact(nivel=W())), salience=10)
    def baja(self):
        self.declare(Fact(nivel="baja"))
```

**Para casa:** explica en 3 líneas por qué se dispara cada regla en cada caso.